# qoo10 top10 추출코드

In [1]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import random
import json
from datetime import datetime

def fetch_qoo10_kbeauty_official_link():
    # 1. 드라이버 설정 (눈으로 확인하기 위해 Headless False)
    options = uc.ChromeOptions()
    # options.add_argument('--headless') 
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--incognito')

    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    options.add_argument(f'user-agent={user_agent}')

    driver = None
    try:
        driver = uc.Chrome(options=options)
        
        # 사용자님이 제공해주신 사이트 자체 검색 링크
        target_url = "https://www.qoo10.jp/s/?keyword_hist=k-beauty&sortType=MOST_REVIEWED&dispType=LIST&curPage=1"

        print(f"📡 큐텐 공식 검색 페이지 접속 중...")
        driver.get(target_url)
        
        # 큐텐은 리스트 로딩에 시간이 걸릴 수 있으므로 넉넉히 대기
        time.sleep(random.uniform(5, 8))

        # 2. 데이터 파싱 시작
        print("📦 페이지 로드 완료, 데이터 추출 시작...")
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # 제공해주신 HTML 구조에 따라 goods 또는 goods_power가 포함된 tr 선택
        items = soup.select('tr[ga-product^="goods"]')
        
        final_rankings = []
        for item in items:
            if len(final_rankings) >= 10: # 상위 10개만 수집
                break
                
            try:
                # A. 상품 코드
                goods_code = item.get('goodscode')

                # B. 브랜드명 (txt_brand 클래스)
                brand_tag = item.select_one("a.txt_brand")
                brand = brand_tag.get_text(strip=True).replace("公式", "").strip() if brand_tag else "N/A"

                # C. 제목 및 상세 URL
                # sbj 내부에서 브랜드 링크가 아닌 진짜 제목 링크 선택
                title_link = item.select_one(".sbj a:not(.txt_brand)")
                if title_link:
                    title = title_link.get('title') or title_link.get_text(strip=True)
                    product_url = title_link.get('href')
                else:
                    continue

                # D. 가격 (숫자만 남기기 위해 정제 로직 추가)
                price_tag = item.select_one(".td_prc .prc strong")
                price_raw = price_tag.get_text(strip=True) if price_tag else "0"
                # '2,200円' -> '2200'
                price_numeric = price_raw.replace('円', '').replace(',', '').strip()

                # E. 평점 (별점 너비%를 5점 만점으로 변환)
                rating_star = item.select_one(".review_rating_star")
                if rating_star and 'style' in rating_star.attrs:
                    width_val = rating_star['style'].split(':')[1].replace('%', '').strip()
                    rating = round(float(width_val) / 20, 2)
                else:
                    rating = 0.0

                # F. 리뷰 개수 (괄호 제거)
                review_tag = item.select_one(".review_total_count")
                reviews_raw = review_tag.get_text(strip=True) if review_tag else "0"
                reviews = reviews_raw.replace('(', '').replace(')', '').replace(',', '').strip()

                final_rankings.append({
                    "rank": len(final_rankings) + 1,
                    "brand": brand,
                    "title": title,
                    "rating": rating,
                    "reviews": int(reviews) if reviews.isdigit() else 0,
                    "price_jpy": int(price_numeric) if price_numeric.isdigit() else 0,
                    "url": product_url,
                    "goods_code": goods_code,
                    "platform": "Qoo10",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                print(f"✅ {len(final_rankings)}위: [{brand}] {rating}점 ({reviews}건)")

            except Exception as e:
                continue

        # 3. JSON 저장
        filename = f"qoo10_kbeauty_top10_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(final_rankings, f, ensure_ascii=False, indent=4)
        
        print(f"\n✨ 수집 및 파일 저장 성공: {filename}")
        return final_rankings

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
    finally:
        if driver:
            time.sleep(5)
            driver.quit()

if __name__ == "__main__":
    fetch_qoo10_kbeauty_official_link()

📡 큐텐 공식 검색 페이지 접속 중...
📦 페이지 로드 완료, 데이터 추출 시작...
✅ 1위: [dear,Klairs] 4.65점 (7275건)
✅ 2위: [dear,Klairs] 4.75점 (3266건)
✅ 3위: [dear,Klairs] 4.75점 (1932건)
✅ 4위: [ホリカホリカ] 4.65점 (1880건)
✅ 5위: [DR.GET IT] 4.65점 (1706건)
✅ 6위: [dear,Klairs] 4.65점 (1386건)
✅ 7위: [dear,Klairs] 4.65점 (1093건)
✅ 8위: [dear,Klairs] 4.65점 (929건)
✅ 9위: [ホリカホリカ] 4.75점 (800건)
✅ 10위: [ホリカホリカ] 4.75점 (712건)

✨ 수집 및 파일 저장 성공: qoo10_kbeauty_top10_20260317_170100.json


# qoo10 단일상품 리뷰데이터 추출코드

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd

def get_qoo10_emergency_fix(gd_no):
    all_reviews = []
    page = 1
    base_url = "https://www.qoo10.jp/gmkt.inc/Goods/GoodsReviewAjaxAppend.aspx"
    
    # 서버 차단을 피하기 위해 브라우저와 거의 동일한 헤더 설정
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "Accept": "text/html, */*",
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8,ko;q=0.7",
        "Referer": f"https://www.qoo10.jp/item/x/{gd_no}", # 실제 상품 페이지 주소를 참조로 넣음
        "X-Requested-With": "XMLHttpRequest" # Ajax 요청임을 명시
    }

    print(f"🚀 Qoo10 상품 {gd_no} 수집 재시도 중...")

    while True:
        params = {
            "gd_no": gd_no,
            "group_code": "2",
            "page_no": page,
            "page_size": "10",
            "sort_type": "P",
            "contents_cnt": "0",
            "___cache_expire___": int(time.time() * 1000)
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers)
            
            # 응답 상태 확인
            if response.status_code != 200:
                print(f"\n❌ 서버 응답 에러 (코드: {response.status_code})")
                break
                
            html_content = response.text.strip()
            if not html_content:
                print(f"\n✅ 수집 완료: 더 이상 읽을 HTML 데이터가 없습니다.")
                break
            
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # 💡 수정된 포인트: li 태그를 찾되, 클래스나 구조에 상관없이 
            # 실제 리뷰 텍스트가 들어있는 영역을 먼저 탐색합니다.
            review_items = soup.find_all('li', recursive=False)
            
            if not review_items:
                # 가끔 상위 태그 없이 li만 올 때가 있으므로 다시 확인
                review_items = soup.select('li')
                
            if not review_items:
                break
                
            for item in review_items:
                # 1. 리뷰 본문 추출 (review_txt 클래스가 핵심)
                txt_tag = item.select_one('.review_txt')
                if not txt_tag: continue # 본문이 없으면 리뷰가 아닐 확률이 높음
                
                content = txt_tag.get_text(strip=True)
                
                # 2. 평점 (score 클래스 또는 스타일 너비)
                score_tag = item.select_one('.score')
                rating = score_tag.get_text(strip=True) if score_tag else "5" # 기본값 5
                
                # 3. 작성자 및 메타정보
                user_info = item.select_one('.review_user_info')
                user_meta = user_info.get_text(" | ", strip=True) if user_info else ""
                
                # 4. 피부 타입/고민
                type_tag = item.select_one('.review_user_type')
                skin_info = type_tag.get_text(strip=True) if type_tag else ""

                all_reviews.append({
                    "Page": page,
                    "Rating": rating,
                    "Review": content,
                    "UserInfo": user_meta,
                    "SkinType": skin_info
                })
            
            print(f"🔄 {page}페이지 수집 중... (누적: {len(all_reviews)}개)", end='\r')
            
            page += 1
            # 💡 차단을 방지하기 위해 지연 시간을 약간 늘립니다.
            time.sleep(1)

        except Exception as e:
            print(f"\n❌ 실행 에러: {e}")
            break
            
    # --- 저장 ---
    output_name = "qoo10_test_crawling.json"
    if all_reviews:
        with open(output_name, 'w', encoding='utf-8') as f:
            json.dump(all_reviews, f, ensure_ascii=False, indent=4)
        print(f"\n📂 수집 성공! 총 {len(all_reviews)}개 저장 완료.")
    
    return pd.DataFrame(all_reviews)

# --- 실행 ---
df = get_qoo10_emergency_fix("501192305")

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
import pandas as pd
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
TARGET_RANK      = 6      # ← 리뷰 수집할 순위
RANK_SAVE_FILE   = "qoo10_kbeauty_rankings.jsonl"
REVIEW_SAVE_FILE = "qoo10_reviews_master.json"


# ── 함수: 리뷰 전체 수집 ─────────────────────────────────────────
def get_qoo10_reviews_all(gd_no, product_name):
    all_reviews = []
    page = 1
    base_url = "https://www.qoo10.jp/gmkt.inc/Goods/GoodsReviewAjaxAppend.aspx"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "Accept": "text/html, */*",
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8,ko;q=0.7",
        "Referer": f"https://www.qoo10.jp/item/x/{gd_no}",
        "X-Requested-With": "XMLHttpRequest"
    }

    print(f"\n🚀 [{product_name}] 전체 리뷰 수집 시작...")

    while True:
        params = {
            "gd_no": gd_no,
            "group_code": "2",
            "page_no": page,
            "page_size": "100",
            "sort_type": "P",
            "contents_cnt": "0",
            "___cache_expire___": int(time.time() * 1000)
        }

        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=15)

            if response.status_code != 200:
                print(f"\n❌ 서버 응답 에러 (코드: {response.status_code})")
                break

            html_content = response.text.strip()
            if not html_content:
                print(f"\n✅ 수집 완료 (총 {len(all_reviews)}개)")
                break

            soup = BeautifulSoup(html_content, 'html.parser')
            review_items = soup.find_all('li', recursive=False) or soup.select('li')

            if not review_items:
                print(f"\n✅ 수집 완료 (총 {len(all_reviews)}개)")
                break

            for item in review_items:
                txt_tag = item.select_one('.review_txt')
                if not txt_tag:
                    continue

                content = txt_tag.get_text(strip=True)
                score_tag = item.select_one('.score')
                rating = score_tag.get_text(strip=True) if score_tag else "5"
                user_info = item.select_one('.review_user_info')
                user_meta = user_info.get_text(" | ", strip=True) if user_info else ""
                type_tag = item.select_one('.review_user_type')
                skin_info = type_tag.get_text(strip=True) if type_tag else ""

                all_reviews.append({
                    "gd_no"   : gd_no,
                    "Page"    : page,
                    "Rating"  : rating,
                    "Review"  : content,
                    "UserInfo": user_meta,
                    "SkinType": skin_info,
                })

            print(f"🔄 {page}페이지 수집 중... (누적: {len(all_reviews)}개)", end='\r')
            page += 1
            time.sleep(1)

        except Exception as e:
            print(f"\n❌ 에러: {e}")
            break

    return all_reviews


# ── STEP 1: Qoo10 Top 10 수집 (Selenium) ─────────────────────────
options = uc.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--incognito')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

driver = None
try:
    driver = uc.Chrome(options=options)
    target_url = "https://www.qoo10.jp/s/?keyword_hist=k-beauty&sortType=MOST_REVIEWED&dispType=LIST&curPage=1"

    print("📡 큐텐 공식 검색 페이지 접속 중...")
    driver.get(target_url)
    time.sleep(random.uniform(5, 8))

    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select('tr[ga-product^="goods"]')

    rank_data_list = []
    review_master  = []
    rank_count     = 1

    for item in items:
        if rank_count > 10:
            break

        try:
            goods_code = item.get('goodscode')

            brand_tag = item.select_one("a.txt_brand")
            brand = brand_tag.get_text(strip=True).replace("公式", "").strip() if brand_tag else "N/A"

            title_link = item.select_one(".sbj a:not(.txt_brand)")
            if not title_link:
                continue
            title = title_link.get('title') or title_link.get_text(strip=True)
            product_url = title_link.get('href')

            price_tag = item.select_one(".td_prc .prc strong")
            price_raw = price_tag.get_text(strip=True) if price_tag else "0"
            price_numeric = price_raw.replace('円', '').replace(',', '').strip()

            rating_star = item.select_one(".review_rating_star")
            if rating_star and 'style' in rating_star.attrs:
                width_val = rating_star['style'].split(':')[1].replace('%', '').strip()
                rating = round(float(width_val) / 20, 2)
            else:
                rating = 0.0

            review_tag = item.select_one(".review_total_count")
            reviews_raw = review_tag.get_text(strip=True) if review_tag else "0"
            reviews = reviews_raw.replace('(', '').replace(')', '').replace(',', '').strip()

            rank_data_list.append({
                "rank"        : rank_count,
                "brand"       : brand,
                "title"       : title,
                "rating"      : rating,
                "reviews"     : int(reviews) if reviews.isdigit() else 0,
                "price_jpy"   : int(price_numeric) if price_numeric.isdigit() else 0,
                "url"         : product_url,
                "goods_code"  : goods_code,
                "platform"    : "Qoo10",
                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            print(f"📍 {rank_count}위 확인: [{brand}] {title[:35]}")

            # ── STEP 2: 6위 상품만 리뷰 전체 수집 ────────────────
            if rank_count == TARGET_RANK:
                review_master = get_qoo10_reviews_all(goods_code, title)

            rank_count += 1

        except Exception as e:
            print(f"⚠️ {rank_count}위 파싱 오류: {e}")
            continue

    # ── STEP 3: 저장 ──────────────────────────────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(review_master, f, ensure_ascii=False, indent=4)

    # ── STEP 4: 결과 출력 ─────────────────────────────────────────
    print()
    print("=" * 65)
    print("🏆 Qoo10 K-Beauty 베스트셀러 Top 10 (리뷰 많은 순)")
    print("=" * 65)
    for item in rank_data_list:
        marker = " ◀ 리뷰 수집됨" if item["rank"] == TARGET_RANK else ""
        print(f"{item['rank']:>2}위 | {item['brand']:<18} | {item['title'][:28]:<28} | ⭐{item['rating']} | ¥{item['price_jpy']}{marker}")
    print("=" * 65)
    print(f"\n📂 저장 완료")
    print(f"  - 순위 파일 : {RANK_SAVE_FILE}")
    print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(review_master)}개)")

    if review_master:
        df = pd.DataFrame(review_master)
        print(f"\n📋 {TARGET_RANK}위 상품 리뷰 상위 5개")
        display(df.head())

finally:
    if driver:
        time.sleep(3)
        driver.quit()

📡 큐텐 공식 검색 페이지 접속 중...
📍 1위 확인: [dear,Klairs] サプルプレパレーションフェイシャルトナー(180ml) / 韓国コスメ
📍 2위 확인: [dear,Klairs] アンセンテッドトナー(180ml) / 韓国コスメ / 化粧水 / 敏
📍 3위 확인: [dear,Klairs] リッチモイストスージングクリーム(80ml) / 韓国コスメ / 水分
📍 4위 확인: [ホリカホリカ] 1+1 MY FAVE PIECE EYE SHADOW //韓国コス
📍 5위 확인: [DR.GET IT] 次世代ダイエットサプリ Kスリム CLA 本格ケア 認定済み 話題のK
📍 6위 확인: [dear,Klairs] ジェントルブラックディープクレンジングオイル(150ml) / メイク

🚀 [ジェントルブラックディープクレンジングオイル(150ml) / メイク落とし / ブラックヘッド] 전체 리뷰 수집 시작...
🔄 14페이지 수집 중... (누적: 1386개)
✅ 수집 완료 (총 1386개)
📍 7위 확인: [dear,Klairs] ブルードロップ(20ml) / ニキビ跡 / 美容液 / 敏感肌 / 
📍 8위 확인: [dear,Klairs] ブルーカーミングクリーム(30ml) / グアイアズレン / 赤み鎮静
📍 9위 확인: [ホリカホリカ] [新色追加] マイフェイブムードアイパレット/韓国コスメ
📍 10위 확인: [ホリカホリカ] ピースマッチング/4色アイシャドウパレット/アイメイク/韓国コスメ

🏆 Qoo10 K-Beauty 베스트셀러 Top 10 (리뷰 많은 순)
 1위 | dear,Klairs        | サプルプレパレーションフェイシャルトナー(180ml)  | ⭐4.65 | ¥2200
 2위 | dear,Klairs        | アンセンテッドトナー(180ml) / 韓国コスメ /  | ⭐4.75 | ¥2200
 3위 | dear,Klairs        | リッチモイストスージングクリーム(80ml) / 韓国コ | ⭐4.75 | ¥2700
 4위 | ホリカホリカ             | 1+1 MY FAVE P

,gd_no,Page,Rating,Review,UserInfo,SkinType
0,498011513,1,5,界面活性剤に頼ってるメイク落ちのいいクレンジングとはちがって、クレアスのはオーガニックオイル...,ai1*** | 2020.02.08 | オプション:D. クレンジングオイル＋クレンジン...,オプション : オプション:D. クレンジングオイル＋クレンジングパフ(+300円)肌タイプ...
1,498011513,1,5,ずーーっと愛用しております。ここ半年でもう少しオイルが厚めで摩擦レスに落とせるクレンジングに...,ka5*** | 2025.12.22 | 商品タイプ:B. クレンジングオイル＋クレンジン...,オプション : 商品タイプ:B. クレンジングオイル＋クレンジングパフ(+300円)肌タイプ...
2,498011513,1,5,やっと、つっぱり感のないクレンジングに出会えました✨乳化すると、もったり重くもっちりな質感に...,aru*** | 2023.04.16 | 商品タイプ:D. クレンジングオイル＋クレンジン...,オプション : 商品タイプ:D. クレンジングオイル＋クレンジングパフ(+300円)肌タイプ...
3,498011513,1,5,クーポンを使って安く購入出来ました。クレンジング１本の購入で、サンプルセットが２つも付いてき...,yuk*** | 2019.07.11 | オプション:B. 現品+Wishtrend10種...,オプション : オプション:B. 現品+Wishtrend10種類サンプル（0702）肌タイ...
4,498011513,1,4,お安く試せました！テクスチャーは軽いかな？と思ったのですが、うまく乳化させるのが結構難しく、...,kiy*** | 2024.01.24 | 商品タイプ:A. [300個限定]クレンジングオイル,オプション : 商品タイプ:A. [300個限定]クレンジングオイル肌タイプ : 混合肌肌...
